# 2.2 自监督学习 (Self-Supervised Learning)从数据自身结构中构造监督信号，无需人工标注，是大模型预训练的核心范式。本节涵盖：- 因果语言建模（CLM）- 掩码语言建模（MLM）- 去噪自编码- 自对比学习

## 1. 因果语言建模（CLM）**目的**：自回归预测下一个token**基本原理**：给定前文token序列，模型预测下一个token的概率分布，使用交叉熵损失优化。

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ftorch.manual_seed(42)class MiniGPT(nn.Module):    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=2):        super().__init__()        self.embedding = nn.Embedding(vocab_size, d_model)        self.pos_embedding = nn.Embedding(512, d_model)        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)        self.lm_head = nn.Linear(d_model, vocab_size)        def forward(self, x):        seq_len = x.size(1)        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)        h = self.embedding(x) + self.pos_embedding(positions)        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)        h = self.transformer(h, mask=causal_mask)        return self.lm_head(h)vocab_size, seq_len, batch_size = 1000, 32, 8model = MiniGPT(vocab_size=vocab_size)input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))logits = model(input_ids)shift_logits = logits[:, :-1, :].contiguous()shift_labels = input_ids[:, 1:].contiguous()loss = F.cross_entropy(shift_logits.view(-1, vocab_size), shift_labels.view(-1))print('=== Causal Language Modeling (CLM) ===')print(f'Input shape: {input_ids.shape}')print(f'Output logits shape: {logits.shape}')print(f'CLM Loss: {loss.item():.4f}')print(f'\nKey: Each position predicts the NEXT token using causal mask.')print(f'Loss computed on shifted: logits[:-1] vs labels[1:]')

## 2. 掩码语言建模（MLM）**目的**：通过预测被遮蔽内容学习双向表征**基本原理**：随机遮蔽输入中15%的token，让模型根据双向上下文预测被遮蔽的内容。

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport randomtorch.manual_seed(42)class MiniBERT(nn.Module):    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=2):        super().__init__()        self.embedding = nn.Embedding(vocab_size, d_model)        self.pos_embedding = nn.Embedding(512, d_model)        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)        self.mlm_head = nn.Linear(d_model, vocab_size)        def forward(self, x):        seq_len = x.size(1)        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)        h = self.embedding(x) + self.pos_embedding(positions)        h = self.transformer(h)        return self.mlm_head(h)def mask_tokens(input_ids, mask_prob=0.15, vocab_size=1000, mask_token_id=999):    labels = input_ids.clone()    probability_matrix = torch.full(labels.shape, mask_prob)    mask = torch.bernoulli(probability_matrix).bool()    labels[~mask] = -100        masked_input = input_ids.clone()    masked_input[mask] = mask_token_id    return masked_input, labels, maskmodel = MiniBERT(vocab_size=1000)mask_token_id = 999input_ids = torch.randint(0, 999, (4, 32))masked_input, labels, mask = mask_tokens(input_ids, mask_prob=0.15, mask_token_id=mask_token_id)logits = model(masked_input)mlm_loss = F.cross_entropy(logits.view(-1, 1000), labels.view(-1), ignore_index=-100)n_masked = mask.sum().item()print('=== Masked Language Modeling (MLM) ===')print(f'Original tokens: {input_ids[0, :10].tolist()}')print(f'Masked tokens:   {masked_input[0, :10].tolist()}')print(f'Mask positions:  {mask[0, :10].tolist()}')print(f'Number of masked tokens: {n_masked}/{input_ids.numel()} ({n_masked/input_ids.numel()*100:.1f}%)')print(f'MLM Loss: {mlm_loss.item():.4f}')print(f'\nKey: BERT-style bidirectional attention sees full context to predict [MASK] tokens.')

## 3. 去噪自编码**目的**：从损坏输入恢复原始文本**基本原理**：对输入施加噪声，训练模型恢复原始文本。

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport randomtorch.manual_seed(42)def add_noise(input_ids, noise_type='random_delete', delete_prob=0.1, vocab_size=1000, sentinel_start=990):    """Add noise to input for denoising autoencoder training."""
    noisy = input_ids.clone()    sentinels = list(range(sentinel_start, sentinel_start + 10))    sentinel_idx = 0        if noise_type == 'random_delete':        keep_mask = torch.bernoulli(torch.full(input_ids.shape, 1 - delete_prob)).bool()        result = []        for i in range(input_ids.size(0)):            kept = noisy[i][keep_mask[i]]            result.append(kept)        return torch.nn.utils.rnn.pad_sequence(result, batch_first=True)        elif noise_type == 'span_delete':        result = noisy.clone()        for i in range(noisy.size(0)):            seq = noisy[i].tolist()            new_seq = []            j = 0            while j < len(seq):                if random.random() < delete_prob and sentinel_idx < len(sentinels):                    span_len = random.randint(1, 3)                    new_seq.append(sentinels[sentinel_idx])                    sentinel_idx += 1                    j += span_len                else:                    new_seq.append(seq[j])                    j += 1            result[i] = torch.tensor(new_seq[:noisy.size(1)], dtype=torch.long)        return resultinput_ids = torch.randint(0, 900, (4, 20))noisy_input = add_noise(input_ids, noise_type='span_delete', delete_prob=0.15)print('=== Denoising Autoencoder ===')print(f'Original:  {input_ids[0, :15].tolist()}')print(f'Noisy:     {noisy_input[0, :15].tolist()}')print(f'\nNoise types: random_delete, span_delete (T5-style), token_replace')print(f'Sentinels (990-999) mark deleted spans; model learns to reconstruct original text.')

## 4. 自对比学习**目的**：从数据增强中学习不变表征**基本原理**：对同一输入进行两次不同的数据增强得到两个视图，训练模型使同一输入的不同视图在表征空间中接近。

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ftorch.manual_seed(42)class ContrastiveEncoder(nn.Module):    def __init__(self, input_dim=64, hidden_dim=128, proj_dim=32):        super().__init__()        self.encoder = nn.Sequential(            nn.Linear(input_dim, hidden_dim),            nn.ReLU(),            nn.Linear(hidden_dim, hidden_dim)        )        self.projector = nn.Sequential(            nn.Linear(hidden_dim, proj_dim),            nn.ReLU(),            nn.Linear(proj_dim, proj_dim)        )        def forward(self, x):        h = self.encoder(x)        z = self.projector(h)        return F.normalize(z, dim=-1)def info_nce_loss(z1, z2, temperature=0.1):    """InfoNCE contrastive loss (NT-Xent).""    batch_size = z1.size(0)    z = torch.cat([z1, z2], dim=0)    sim = torch.mm(z, z.t()) / temperature        mask = torch.eye(2 * batch_size, dtype=torch.bool)    sim.masked_fill_(mask, -1e9)        labels = torch.cat([torch.arange(batch_size, 2*batch_size), torch.arange(0, batch_size)])    loss = F.cross_entropy(sim, labels)    return lossdef augment(x, noise_std=0.1):    return x + torch.randn_like(x) * noise_stdencoder = ContrastiveEncoder()optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)n_samples = 200X = torch.randn(n_samples, 64)print('=== Self-Supervised Contrastive Learning ===')for epoch in range(30):    x1 = augment(X)    x2 = augment(X)    z1, z2 = encoder(x1), encoder(x2)    loss = info_nce_loss(z1, z2)        optimizer.zero_grad()    loss.backward()    optimizer.step()        if (epoch + 1) % 10 == 0:        with torch.no_grad():            sim_pos = F.cosine_similarity(z1, z2).mean()            sim_neg = F.cosine_similarity(z1[:5], z2[5:10]).mean()        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, pos_sim={sim_pos:.3f}, neg_sim={sim_neg:.3f}')print('\nKey: Positive pairs (augmented views of same input) are pulled together,')print('negative pairs (different inputs) are pushed apart via InfoNCE loss.')